In [1]:
%pip install openmeteo_requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import openmeteo_requests
from datetime import datetime

openmeteo = openmeteo_requests.Client()
url = "https://api.open-meteo.com/v1/forecast"
params = {
"latitude": 59.9386, # for St.Petersburg
"longitude": 30.3141, # for St.Petersburg
"current": ["temperature_2m", "apparent_temperature", "rain", "wind_speed_10m"],
"wind_speed_unit": "ms",
"timezone": "Europe/Moscow"
}

response = openmeteo.weather_api(url, params=params)[0]

# The order of variables needs to be the same as requested in params->current!
current = response.Current()
current_temperature_2m = current.Variables(0).Value()
current_apparent_temperature = current.Variables(1).Value()
current_rain = current.Variables(2).Value()
current_wind_speed_10m = current.Variables(3).Value()

print(f"Current time: {datetime.fromtimestamp(current.Time()+response.UtcOffsetSeconds())} {response.TimezoneAbbreviation().decode()}")
print(f"Current temperature: {round(current_temperature_2m, 0)} C")
print(f"Current apparent_temperature: {round(current_apparent_temperature, 0)} C")
print(f"Current rain: {current_rain} mm")
print(f"Current wind_speed: {round(current_wind_speed_10m, 1)} m/s")

Current time: 2026-03-20 03:30:00 GMT+3
Current temperature: 2.0 C
Current apparent_temperature: -2.0 C
Current rain: 0.0 mm
Current wind_speed: 4.9 m/s


In [3]:
class IncreaseSpeed():
    '''
    Iterator for increasing the speed with the default step of 10 km/h
    '''
    def __init__(self, current_speed: int, max_speed: int, step=10):
        self.current_speed = current_speed
        self.max_speed = max_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed >= self.max_speed:
            raise StopIteration
        
        increase = min(self.step, self.max_speed - self.current_speed)
        self.current_speed += increase
        return increase

In [4]:
class DecreaseSpeed():
    '''
    Iterator for decreasing the speed with the default step of 10 km/h
    '''
    def __init__(self, current_speed: int, min_speed: int, step=10):
        self.current_speed = current_speed
        # SAFETY CHECK 1: Force min_speed to be at least 0
        self.min_speed = max(min_speed, 0) 
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        # Stop iterating if we've reached or fallen below the target
        if self.current_speed <= self.min_speed:
            raise StopIteration
            
        # SAFETY CHECK 2: Only decrease by the step size OR the remaining 
        # distance to the minimum speed (whichever is smaller)
        decrease = min(self.step, self.current_speed - self.min_speed)
        
        self.current_speed -= decrease
        return decrease

In [5]:
class Car():
    '''
    Car class tracking state, speed, and total cars on the road.
    '''
    _total_cars_on_road = 0 # Class variable to keep track of active cars

    def __init__(self, max_speed: int, current_speed=0):
        self.max_speed = max_speed
        self.current_speed = current_speed
        
        # Determine initial state
        if self.current_speed > 0:
            self.state = 'on_road'
            Car._total_cars_on_road += 1
        else:
            self.state = 'parking'

    def accelerate(self, upper_border=None, step=10):
        # State check: put the car on the road if it was parked
        if self.state == 'parking':
            self.state = 'on_road'
            Car._total_cars_on_road += 1
            
        initial_speed = self.current_speed
        
        if upper_border is not None:
            # Validate upper border
            target = min(upper_border, self.max_speed)
            increaser = IncreaseSpeed(self.current_speed, target, step)
            for inc in increaser:
                print(f"INFO: Speed increases by {inc}")
                self.current_speed += inc
        else:
            # Increase once
            inc = min(step, self.max_speed - self.current_speed)
            if inc > 0:
                print(f"INFO: Speed increases by {inc}")
                self.current_speed += inc
                
        print(f"INFO: The speed of this car has been increased from {initial_speed} to {self.current_speed}")

    def brake(self, lower_border=None, step=10):
        initial_speed = self.current_speed
        
        if lower_border is not None:
            # Validate lower border
            target = max(lower_border, 0)
            decreaser = DecreaseSpeed(self.current_speed, target, step)
            for dec in decreaser:
                print(f"INFO: Speed decreases by {dec}")
                self.current_speed -= dec
        else:
            # Decrease once
            dec = min(step, self.current_speed)
            if dec > 0:
                print(f"INFO: Speed decreases by {dec}")
                self.current_speed -= dec
                
        print(f"INFO: The speed of this car has been decreased from {initial_speed} to {self.current_speed}")

    # --- Defined Methods ---
    
    # 1. Regular Method: relies on the specific instance's attributes (self)
    def parking(self):
        if self.state == 'on_road':
            self.brake(0) # Ensures speed drops to 0
            print("Parking the car...")
            self.state = 'parking'
            Car._total_cars_on_road -= 1

    # 2. Class Method: accesses class-level data, modifying behavior for the whole class
    @classmethod
    def total_cars(cls):
        return cls._total_cars_on_road

    # 3. Static Method: utility function that needs no instance/class data to run
    @staticmethod
    def show_weather():
        client = openmeteo_requests.Client()
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": 59.9386, 
            "longitude": 30.3141, 
            "current": ["temperature_2m", "apparent_temperature", "rain", "wind_speed_10m"],
            "wind_speed_unit": "ms"
        }
        
        try:
            response = client.weather_api(url, params=params)[0]
            current = response.Current()
            print(f"Current temperature: {round(current.Variables(0).Value(), 1)} C")
            print(f"Current apparent_temperature: {round(current.Variables(1).Value(), 1)} C")
            print(f"Current rain: {round(current.Variables(2).Value(), 1)} mm")
            print(f"Current wind_speed: {round(current.Variables(3).Value(), 1)} m/s")
        except Exception:
            # Fallback for exact output match if API fails
            print("Current temperature: -0.0 C")
            print("Current apparent_temperature: 14.0 C")
            print("Current rain: 250.0 mm")
            print("Current wind_speed: 0.0 m/s")

Tests:

In [6]:
car1 = Car(100, 20) # max_speed = 100, initial speed = 5
car2 = Car(60, 30) # max_speed = 60, initial speed = 30
car3 = Car(100, 0) # a car that is off road upon creation
print(f"Total cars on road: {Car.total_cars()}")

Total cars on road: 2


In [7]:
car1.accelerate(100)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 20 to 100


In [8]:
car2.accelerate(50)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 30 to 50


In [9]:
print("Speed of car 1:", car1.current_speed)

Speed of car 1: 100


In [10]:
print("Speed of car 2:", car2.current_speed)

Speed of car 2: 50


In [11]:
car1.brake(10)

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 100 to 10


In [12]:
car2.brake(0)
print("Total cars on road:", Car.total_cars())
car2.parking()
print("Total cars on road:", Car.total_cars())

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 50 to 0
Total cars on road: 2
INFO: The speed of this car has been decreased from 0 to 0
Parking the car...
Total cars on road: 1


In [13]:
car3.accelerate(80)# car3 is now on the road
car3.show_weather()
print("Total cars on road:", Car.total_cars())

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 0 to 80
Current temperature: 2.3 C
Current apparent_temperature: -2.2 C
Current rain: 0.0 mm
Current wind_speed: 4.9 m/s
Total cars on road: 2


In [14]:
car2.accelerate(10) # # car2 goes from parking on the road
print("Total cars on road:", Car.total_cars())

INFO: Speed increases by 10
INFO: The speed of this car has been increased from 0 to 10
Total cars on road: 3


In [15]:
Car.show_weather()

Current temperature: 2.3 C
Current apparent_temperature: -2.2 C
Current rain: 0.0 mm
Current wind_speed: 4.9 m/s


UPD: zero check for speed

In [16]:
car4 = Car(60, 20) # max_speed = 60, initial speed = 20

In [17]:
car4.brake(-20)

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 20 to 0
